# Модель выявления аномалий (Isolation Forest)

**Задача:** обнаружить паттерны ошибочных записей, которые не покрываются ручными правилами.

**Подход:**
1. Загружаем очищенный датасет (после пайплайна нормализации и точечных фиксов)
2. Извлекаем числовые признаки из каждой записи
3. Обучаем Isolation Forest — unsupervised модель для обнаружения аномалий
4. Сравниваем результаты модели с правилами: что нового нашла модель?
5. Интерпретируем паттерны

## 1. Загрузка очищенного датасета

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Запускаем пайплайн очистки (из pipeline_v3.py)
# Если данные уже очищены — можно загрузить clean_final.csv
# df = pd.read_csv('clean_final.csv', parse_dates=['bdate','test_date','guard_bdate'])

# --- Полный пайплайн ---
df = pd.read_csv('data/hakaton.csv', sep=';')
df['result'] = df['result'].str.strip().str.upper()

cols_content = [c for c in df.columns if c not in ['our_number', 'name_naprav', 'name_area']]
df = df.drop_duplicates(subset=cols_content, keep='first')
df = df.drop(columns=['our_number', 'name_naprav', 'name_area'])

for col in ['bdate', 'test_date', 'guard_bdate']:
    df[col] = pd.to_datetime(df[col])

df['id_doc'] = df['id_doc'].replace('-', np.nan)

def normalize_id(v):
    if pd.isna(v): return v
    cleaned = re.sub(r'[№N\-\s]', '', str(v).strip())
    return cleaned if (cleaned and any(c.isdigit() for c in cleaned)) else np.nan

df['id_doc'] = df['id_doc'].apply(normalize_id)
df['guard_id_doc'] = df['guard_id_doc'].apply(normalize_id)

def normalize_variant(v):
    if pd.isna(v): return v
    v = str(v).strip()
    try:
        int(v); return v
    except: pass
    for pattern, repl in [
        (r'^УЧ\s+(\d+)', lambda m: m.group(1)),
        (r'^7[24]7[._/](\d{5,7})$', lambda m: m.group(1)),
        (r'^727\.(\d+)\.(\d+)$', lambda m: m.group(1)+m.group(2)),
        (r'^№\s*(\d+)', lambda m: m.group(1)),
        (r'^0\s+(\d+)$', lambda m: '0'+m.group(1)),
        (r'^`(\d+)$', lambda m: m.group(1)),
        (r'^(\d{3,})\.(\d+)$', lambda m: m.group(1)+m.group(2)),
    ]:
        m = re.match(pattern, v)
        if m: return repl(m)
    return np.nan

df['variant'] = df['variant'].apply(normalize_variant)
df.loc[df['class'] == '2-5', 'class'] = '3'

mask_valid = df['id_doc'].notna()
df_v = df[mask_valid].copy().sort_values(['id_doc','test_date','variant','class','result'])
df_v = df_v.drop_duplicates(subset=['id_doc','test_date','variant','class'], keep='last')
df = pd.concat([df_v, df[~mask_valid]], ignore_index=True)

# Точечные фиксы
df = df[df['ogrn_naprav'].astype(str).str.len() == 13]
df = df[df['variant'].notna()]
for id_doc, old, new in [('13653549','2025-09-18','2012-09-18'),('0384771','2025-09-14','2018-09-14'),
                          ('18151280','2025-05-16','2015-05-16'),('405348032','2025-08-13','2011-08-13'),
                          ('0535794','2006-01-23','2011-01-23'),('1306476','1996-10-04','2009-10-04'),
                          ('576809','1992-02-27','2018-02-27')]:
    df.loc[(df['id_doc']==id_doc)&(df['bdate']==pd.Timestamp(old)),'bdate'] = pd.Timestamp(new)

df['child_age'] = (df['test_date']-df['bdate']).dt.days/365.25
df = df[~(df['child_age']>20)]
df = df[~((df['id_doc']=='15238017')&(df['bdate'].dt.year==2023))]
df = df.drop(columns=['child_age']).reset_index(drop=True)

# Заполняем пустые id_doc
df['person_key'] = df['last_name'].str.strip()+'|'+df['first_name'].str.strip()+'|'+df['middle_name'].fillna('').str.strip()+'|'+df['bdate'].dt.strftime('%Y-%m-%d')
has_id = df[df['id_doc'].notna()]
id_map = has_id.groupby('person_key')['id_doc'].agg(lambda x: x.mode().iloc[0])
for idx in df[df['id_doc'].isna()].index:
    key = df.at[idx,'person_key']
    if key in id_map.index: df.at[idx,'id_doc'] = id_map[key]
still_na = df['id_doc'].isna()
if still_na.any():
    counter = int(pd.to_numeric(df['id_doc'].dropna(), errors='coerce').max()) + 1
    for key in df.loc[still_na,'person_key'].unique():
        df.loc[still_na & (df['person_key']==key), 'id_doc'] = str(counter)
        counter += 1

id_map_full = df.groupby('person_key')['id_doc'].agg(lambda x: x.mode().iloc[0])
df['person_id'] = df['person_key'].map(id_map_full)

print(f"Датасет: {len(df)} записей, {df['person_id'].nunique()} уникальных детей")

## 2. Feature Engineering

Для каждой записи извлекаем числовые признаки, которые описывают «нормальность» записи:

**Признаки записи:**
- `child_age` — возраст ребёнка на момент теста
- `parent_age_at_birth` — сколько лет было родителю при рождении ребёнка
- `age_class_diff` — отклонение возраста от ожидаемого для класса (класс + 6)
- `id_eq_guard` — совпадает ли id_doc с guard_id_doc
- `same_school` — совпадает ли направляющая школа с площадкой тестирования
- `class_num` — номер класса

**Агрегаты по ребёнку (person_id):**
- `n_tests` — сколько раз тестировался
- `n_classes` — сколько разных классов
- `min_gap` — минимальный интервал между тестами (дней)
- `max_class_jump` — максимальный скачок класса между тестами

In [ ]:
# === Признаки записи ===
df['class_num'] = pd.to_numeric(df['class'], errors='coerce')
df['child_age'] = (df['test_date'] - df['bdate']).dt.days / 365.25
df['parent_age_at_birth'] = (df['bdate'] - df['guard_bdate']).dt.days / 365.25
df['age_class_diff'] = df['child_age'] - (df['class_num'] + 6)
df['id_eq_guard'] = (df['id_doc'] == df['guard_id_doc']).astype(int)
df['same_school'] = (df['ogrn_naprav'] == df['ogrn_area']).astype(int)

# === Агрегаты по person_id ===
person_stats = df.groupby('person_id').agg(
    n_tests=('test_date', 'count'),
    n_classes=('class_num', 'nunique'),
).reset_index()

# Минимальный интервал между тестами
df_sorted = df.sort_values(['person_id', 'test_date'])
df_sorted['days_gap'] = (
    df_sorted['test_date'] - df_sorted.groupby('person_id')['test_date'].shift(1)
).dt.days

gap_stats = df_sorted.groupby('person_id')['days_gap'].agg(min_gap='min').reset_index()
gap_stats['min_gap'] = gap_stats['min_gap'].fillna(999)  # нет повторных тестов

# Максимальный скачок класса
def max_class_jump(group):
    classes = group.sort_values('test_date')['class_num'].dropna().values
    if len(classes) < 2:
        return 0
    return max(abs(classes[i] - classes[i-1]) for i in range(1, len(classes)))

jump_stats = df.groupby('person_id').apply(max_class_jump).reset_index()
jump_stats.columns = ['person_id', 'max_class_jump']

# Мёржим агрегаты
df = df.merge(person_stats, on='person_id', how='left')
df = df.merge(gap_stats, on='person_id', how='left')
df = df.merge(jump_stats, on='person_id', how='left')

# Итоговый набор фичей
feature_cols = [
    'child_age', 'parent_age_at_birth', 'age_class_diff',
    'id_eq_guard', 'same_school',
    'n_tests', 'n_classes', 'min_gap', 'max_class_jump', 'class_num'
]

X = df[feature_cols].fillna(0)
print(f"Матрица признаков: {X.shape}")
print(f"\nСтатистика:")
print(X.describe().round(2))

## 3. Правила (baseline)

Прежде чем запускать модель, пересчитаем флаги аномалий по правилам — это наш baseline для сравнения.

In [ ]:
# Правила из пайплайна очистки
df['rule_freq'] = df['min_gap'] < 90                     # частота < 90 дней
df['rule_multi_class'] = df['max_class_jump'] > 1         # скачок класса > 1
df['rule_id_guard'] = df['id_eq_guard'] == 1              # id_doc == guard_id_doc
df['rule_age_mismatch'] = df['age_class_diff'].abs() > 3  # возраст-класс > 3 года
df['rule_guard_young'] = (df['parent_age_at_birth'] > 0) & (df['parent_age_at_birth'] < 14)

rule_cols = ['rule_freq', 'rule_multi_class', 'rule_id_guard', 'rule_age_mismatch', 'rule_guard_young']
df['rule_anomaly'] = df[rule_cols].any(axis=1)

print("Аномалии по правилам:")
for col in rule_cols:
    print(f"  {col}: {df[col].sum()}")
print(f"\n  Всего (любое правило): {df['rule_anomaly'].sum()} ({df['rule_anomaly'].mean()*100:.1f}%)")

## 4. Isolation Forest

Обучаем модель на всех записях. Isolation Forest не требует разметки — он находит точки, которые «легко изолировать» от остальных, т.е. выбросы.

Параметр `contamination=0.08` — ожидаемая доля аномалий (~8%, как у наших правил).

In [ ]:
# Масштабирование фичей
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Обучение модели
model = IsolationForest(
    n_estimators=200,
    contamination=0.08,
    random_state=42
)
model.fit(X_scaled)

# Результаты
df['anomaly_score'] = model.decision_function(X_scaled)  # чем ниже — тем аномальнее
df['model_anomaly'] = model.predict(X_scaled) == -1       # True = аномалия

print(f"Anomaly score: min={df['anomaly_score'].min():.3f}, max={df['anomaly_score'].max():.3f}")
print(f"Аномалий по модели: {df['model_anomaly'].sum()} ({df['model_anomaly'].mean()*100:.1f}%)")

## 5. Сравнение: модель vs правила

Ключевой вопрос — что модель нашла **сверх** правил?

In [ ]:
both = (df['model_anomaly'] & df['rule_anomaly']).sum()
model_only = (df['model_anomaly'] & ~df['rule_anomaly']).sum()
rule_only = (~df['model_anomaly'] & df['rule_anomaly']).sum()
neither = (~df['model_anomaly'] & ~df['rule_anomaly']).sum()

print(f"{'':20} | Модель: ДА | Модель: НЕТ")
print(f"{'-'*50}")
print(f"{'Правила: ДА':20} | {both:10} | {rule_only:11}")
print(f"{'Правила: НЕТ':20} | {model_only:10} | {neither:11}")
print(f"")
print(f"Совпадение (оба):    {both}")
print(f"Только модель:       {model_only}  ← НОВЫЕ НАХОДКИ")
print(f"Только правила:      {rule_only}")
print(f"Чистые (оба):        {neither}")

## 6. Что модель нашла нового?

Анализируем записи, которые модель пометила как аномальные, но правила пропустили.

In [ ]:
new_finds = df[df['model_anomaly'] & ~df['rule_anomaly']].copy()
all_data = df.copy()

print(f"Записей 'только модель': {len(new_finds)}")
print(f"\n{'='*60}")
print("СРАВНЕНИЕ СРЕДНИХ: 'только модель' vs весь датасет")
print(f"{'='*60}")
print(f"{'Признак':25} | {'Модель':>10} | {'Всё':>10} | {'Отклонение':>10}")
print(f"{'-'*60}")

for col in feature_cols:
    m_new = new_finds[col].mean()
    m_all = all_data[col].mean()
    diff = m_new - m_all
    marker = ' ←←←' if abs(diff) > 1 else ''
    print(f"{col:25} | {m_new:10.2f} | {m_all:10.2f} | {diff:+10.2f}{marker}")

In [ ]:
# Ключевая находка: same_school
print("\n" + "="*60)
print("КЛЮЧЕВАЯ НАХОДКА: same_school (ogrn_naprav == ogrn_area)")
print("="*60)

pct_new = new_finds['same_school'].mean() * 100
pct_all = all_data['same_school'].mean() * 100

print(f"\n  Доля same_school в 'только модель': {pct_new:.1f}%")
print(f"  Доля same_school во всём датасете:   {pct_all:.1f}%")
print(f"  Превышение: в {pct_new/pct_all:.1f} раз")
print(f"")
print(f"  Это значит: школа НАПРАВЛЯЕТ ребёнка на тестирование")
print(f"  и САМА ЖЕ проводит это тестирование.")
print(f"  Потенциальный конфликт интересов / нарушение процедуры.")

In [ ]:
# Распределение классов в находках модели
print("\nРаспределение по классам ('только модель'):")
print(new_finds['class_num'].value_counts().sort_index().to_string())
print(f"\nСредний класс: {new_finds['class_num'].mean():.1f} vs общий: {all_data['class_num'].mean():.1f}")
print(f"Средний возраст: {new_finds['child_age'].mean():.1f} vs общий: {all_data['child_age'].mean():.1f}")

## 7. Важность признаков

Оценим, какие признаки больше всего влияют на решение модели, сравнив средние anomaly_score по значениям каждого признака.

In [ ]:
# Корреляция фичей с anomaly_score
print("Корреляция признаков с anomaly_score:")
print("(отрицательная = чем больше признак, тем аномальнее)")
print()

correlations = df[feature_cols + ['anomaly_score']].corr()['anomaly_score'].drop('anomaly_score')
for feat, corr in correlations.abs().sort_values(ascending=False).items():
    sign = correlations[feat]
    direction = "↑ больше = аномальнее" if sign < 0 else "↑ больше = нормальнее"
    print(f"  {feat:25} r={correlations[feat]:+.3f}  {direction}")

## 8. Топ самых аномальных записей по модели

In [ ]:
top_anomalies = df.nsmallest(20, 'anomaly_score')
display_cols = ['last_name', 'first_name', 'class', 'test_date', 'anomaly_score', 
                'rule_anomaly'] + feature_cols

print("Топ-20 самых аномальных записей (по anomaly_score):")
print(top_anomalies[display_cols].to_string())

## 9. Итого и сохранение

**Что дала модель поверх правил:**
- Обнаружен паттерн `same_school` — направляющая школа совпадает с площадкой тестирования, что может свидетельствовать о нарушении процедуры
- Модель чаще флагает старшеклассников (8-11 класс) — возможно, в этих классах больше процедурных нарушений
- Anomaly score позволяет ранжировать записи по степени подозрительности, а не просто бинарный флаг

In [ ]:
# Объединённый флаг: правила ИЛИ модель
df['combined_anomaly'] = df['rule_anomaly'] | df['model_anomaly']

print(f"Итоговая статистика:")
print(f"  Только правила:            {df['rule_anomaly'].sum()}")
print(f"  Только модель:             {df['model_anomaly'].sum()}")
print(f"  Объединённый флаг:         {df['combined_anomaly'].sum()} ({df['combined_anomaly'].mean()*100:.1f}%)")
print(f"  Чистых записей:            {(~df['combined_anomaly']).sum()}")

# Сохраняем
export = df.drop(columns=['person_key'], errors='ignore')
export.to_csv('dataset_with_model_scores.csv', index=False)
print(f"\nСохранено: dataset_with_model_scores.csv")